# HR Employee Attrition — Exploratory Data Analysis & Predictive Modeling
End-to-end analysis of `WA_Fn-UseC_-HR-Employee-Attrition.csv` (1,470 employee records).

In [ ]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_preprocessing import load_and_clean_data, engineer_features

df = engineer_features(load_and_clean_data('../data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv'))
print('Shape:', df.shape)
df.head()

## 1. Target Distribution & Key Segment Breakdowns

In [ ]:
print('Overall Attrition Rate (%):')
print((df['Attrition'].value_counts(normalize=True) * 100).round(2))

print('\nAttrition by OverTime (%):')
print((pd.crosstab(df['OverTime'], df['Attrition'], normalize='index') * 100).round(2))

print('\nAttrition by JobRole (%):')
print((pd.crosstab(df['JobRole'], df['Attrition'], normalize='index') * 100).sort_values('Yes', ascending=False).round(2))

## 2. Train & Evaluate Class-Weighted Pipelines

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score
from src.train import build_preprocessor

X = df.drop(columns=['Attrition'])
y = (df['Attrition'] == 'Yes').astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
pipe = Pipeline([('prep', build_preprocessor(X)), ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))])
pipe.fit(X_train, y_train)

probs = pipe.predict_proba(X_test)[:, 1]
preds = pipe.predict(X_test)
print(f'Test ROC-AUC: {roc_auc_score(y_test, probs):.4f}')
print(classification_report(y_test, preds))